# Импорт и установка необходимых инструментов

In [ ]:
import sys
!{sys.executable} -m pip install tifffile matplotlib scipy bm3d scikit-image

import os
import shutil
from pathlib import Path
import numpy as np
import tifffile
import matplotlib.pyplot as plt
import cv2
from skimage.restoration import estimate_sigma
import bm3d
from skimage.color import rgb2gray
from skimage.restoration import (
    denoise_nl_means,
    denoise_wavelet,
    denoise_bilateral
)

# Обнуление красного канала

In [ ]:
def zero_red_channel(img):
    """Обнуляет красный канал (индекс 0) в RGB-изображении."""
    if img.ndim == 3 and img.shape[-1] >= 3:
        img_copy = img.copy()
        img_copy[..., 0] = 0
        return img_copy
    return img

source_root = r"D:\astrocytes data\data"          
target_root = r"D:\astrocytes data\data_zero_red"  
image_ext = {'.tif', '.tiff'}               

total_processed = 0

for root, dirs, files in os.walk(source_root):
    rel_path = os.path.relpath(root, source_root)
    target_dir = os.path.join(target_root, rel_path)
    
    # Собираем все TIF-файлы в текущей папке
    tif_files = [f for f in files if Path(f).suffix.lower() in image_ext]
    
    if not tif_files:
        os.makedirs(target_dir, exist_ok=True)
        continue
    
    os.makedirs(target_dir, exist_ok=True)
    
    for fname in tif_files:
        src_path = os.path.join(root, fname)
        dst_path = os.path.join(target_dir, fname)
        try:
            img = tifffile.imread(src_path)
            img_zero = zero_red_channel(img)
            tifffile.imwrite(dst_path, img_zero)
            total_processed += 1
        except Exception as e:
            print(f"Ошибка при обработке {src_path}: {e}")
    
    print(f"{rel_path or 'корень'}: обработано {len(tif_files)} файлов")

print(f"\nГотово! Обработано файлов: {total_processed}")
print(f"Результаты сохранены в: {target_root}")

# Перевод изображений в черно-белый формат


In [ ]:
def to_grayscale(img):
    """
    Преобразует RGB-изображение в grayscale (один канал).
    """
    if img.ndim == 3 and img.shape[-1] == 3:
        if img.dtype == np.uint16:
            img_float = img.astype(np.float32)
            gray = 0.299 * img_float[..., 0] + 0.587 * img_float[..., 1] + 0.114 * img_float[..., 2]
            gray = np.round(gray).astype(np.uint16)
        elif img.dtype == np.uint8:
            gray = (0.299 * img[..., 0] + 0.587 * img[..., 1] + 0.114 * img[..., 2]).astype(np.uint8)
        else:
            gray = 0.299 * img[..., 0] + 0.587 * img[..., 1] + 0.114 * img[..., 2]
            gray = gray.astype(img.dtype)
        return gray
    else:
        return img


source_root = r"D:\astrocytes data\data"         
target_root = r"D:\astrocytes data\data_BW"
image_ext = {'.tif', '.tiff'}

total_processed = 0

for root, dirs, files in os.walk(source_root):
    rel_path = os.path.relpath(root, source_root)
    target_dir = os.path.join(target_root, rel_path)
    
    tif_files = [f for f in files if Path(f).suffix.lower() in image_ext]
    if not tif_files:
        os.makedirs(target_dir, exist_ok=True)
        continue
    
    os.makedirs(target_dir, exist_ok=True)
    
    for fname in tif_files:
        src_path = os.path.join(root, fname)
        dst_path = os.path.join(target_dir, fname)
        try:
            img = tifffile.imread(src_path)
            img_gray = to_grayscale(img)
            tifffile.imwrite(dst_path, img_gray)
            total_processed += 1
        except Exception as e:
            print(f"Ошибка при обработке {src_path}: {e}")
    
    print(f"{rel_path or 'корень'}: обработано {len(tif_files)} файлов")

print(f"\n Готово! Обработано файлов: {total_processed}")
print(f"Результаты сохранены в: {target_root}")

# Применение BM3d-фильтра

In [ ]:

input_base = r"D:\astrocytes data\data"      
output_base = r"D:\astrocytes data\data+BM3D"   

def process_tiff_files(input_folder, output_folder):
    
    for root, dirs, files in os.walk(input_folder):
        rel_path = os.path.relpath(root, input_folder)
        current_output_dir = os.path.join(output_folder, rel_path)
        os.makedirs(current_output_dir, exist_ok=True)

        for filename in files:
            if filename.lower().endswith(('.tiff', '.tif')):
                input_path = os.path.join(root, filename)
                output_filename = f'denoised_{filename}'
                output_path = os.path.join(current_output_dir, output_filename)

                if os.path.exists(output_path):
                    print(f'Файл уже обработан: {output_path}')
                    continue

                print(f'Обработка файла: {input_path}')
                try:
                    img = tifffile.imread(input_path)

                    if img.ndim == 3 and img.shape[-1] == 3:
                        img = rgb2gray(img)
                        img = (img * 65535).astype(np.uint16)

                    if img.ndim != 2:
                        print(f"Пропускаем файл {input_path}: не поддерживаемая размерность {img.shape}")
                        continue

                    img_float = img.astype(np.float32) / 65535.0

                    sigma_estimated = estimate_sigma(img_float, average_sigmas=True)

                    sigma_psd = sigma_estimated
                    
                    img_denoised = bm3d.bm3d(
                            img_float,
                            sigma_psd=sigma_psd,
                            stage_arg=bm3d.BM3DStages.ALL_STAGES
                    )

                    
                    img_denoised = np.clip(img_denoised * 65535, 0, 65535).astype(np.uint16)

                    tifffile.imwrite(output_path, img_denoised)
                    print(f'Сохранено: {output_path}')

                except Exception as e:
                    print(f"Ошибка при обработке {input_path}: {e}")

process_tiff_files(input_base, output_base)

# Применение ансамбля фильтров - NLM, Wavelet, Bilateral

In [ ]:

input_base = r"D:\astrocytes data\data"
output_base = r"D:\astrocytes data\data+ensemble"

def ensemble_parallel(img_float, sigma_psd):
    """Параллельный ансамбль (усреднение результатов трёх фильтров)"""
    nlm = denoise_nl_means(
        img_float,
        h=1.5 * sigma_psd,
        patch_size=7,
        patch_distance=11,
        fast_mode=True,
        preserve_range=True
    )
    wav = denoise_wavelet(
        img_float,
        sigma=sigma_psd,
        method='BayesShrink',
        wavelet='sym4'
    )
    bil = denoise_bilateral(
        img_float,
        sigma_color=1.5 * sigma_psd,
        sigma_spatial=2,
        win_size=5
    )
    return (nlm + wav + bil) / 3.0


def process_tiff_files(input_folder, output_folder):
    for root, dirs, files in os.walk(input_folder):
        rel_path = os.path.relpath(root, input_folder)
        current_output_dir = os.path.join(output_folder, rel_path)
        os.makedirs(current_output_dir, exist_ok=True)

        for filename in files:
            if not filename.lower().endswith(('.tiff', '.tif')):
                continue

            input_path = os.path.join(root, filename)
            output_filename = f'denoised_{filename}'
            output_path = os.path.join(current_output_dir, output_filename)

            if os.path.exists(output_path):
                print(f'Файл уже обработан: {output_path}')
                continue

            print(f'Обработка файла: {input_path}')
            try:
                img = tifffile.imread(input_path)
                
                if img.ndim == 3 and img.shape[-1] == 3:
                    img = (rgb2gray(img) * 65535).astype(np.uint16)

                elif img.ndim == 3:
                    img = img[:, :, 0]

                if img.ndim != 2:
                    print(f"Пропускаем {input_path}: размерность {img.shape}")
                    continue

                img_float = img.astype(np.float32) / 65535.0

                sigma_psd = estimate_sigma(img_float, average_sigmas=True)
                print(f"  Оценка sigma = {sigma_psd:.4f}")

                img_denoised = ensemble_parallel(img_float, sigma_psd)

                img_denoised = np.clip(img_denoised * 65535, 0, 65535).astype(np.uint16)

                tifffile.imwrite(output_path, img_denoised)
                print(f'Сохранено: {output_path}')

            except Exception as e:
                print(f"Ошибка при обработке {input_path}: {e}")

process_tiff_files(input_base, output_base)